# Sprint 3 — Final Reusable Outcome-Prediction Pipeline

This notebook demonstrates the final end-to-end implementation of the project.

The actual pipeline logic is implemented in reusable Python modules:

- `src/pipeline/training_pipeline.py`
- `src/pipeline/evaluation_pipeline.py`

The notebook only configures, runs, and presents the pipeline. It does not duplicate data preparation, model training, evaluation, plotting, or SHAP logic.

The pipeline performs:

1. event-log loading;
2. final-activity outcome construction;
3. temporal train/validation/test splitting;
4. prefix generation after splitting;
5. aligned feature encoding;
6. leakage-related feature removal;
7. training of a majority baseline, Logistic Regression, and Random Forest;
8. validation-based Random Forest selection;
9. final test-set evaluation;
10. prediction export, plots, and SHAP explanations.


## 1. Project setup

The following cell locates the repository root. It works when the notebook is started either from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

current_directory = Path.cwd().resolve()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## 2. Import the reusable pipelines

Only the public pipeline functions are imported here. All implementation details remain inside `src/`.


In [ ]:
from src.pipeline import (
    run_evaluation_pipeline,
    run_training_pipeline,
)

## 3. Configuration

The final outcome is based on the final lifecycle transition of each original case:

- **positive outcome (`1`)**: the case ends in `Closed` or `Resolved`;
- **negative outcome (`0`)**: the case ends in another final transition.

The original cases are split temporally before prefixes are generated. This prevents prefixes from one original case from appearing in multiple data splits.


In [ ]:
from scripts.download_data import download_dataset

# Download the event log on first run (idempotent); returns the .xes path.
data_path = download_dataset()

output_dir = project_root / "outputs"

positive_final_activities = ["Closed", "Resolved"]
minimum_prefix_length = 1

print(f"Data path: {data_path}")
print(f"Output directory: {output_dir}")
print(f"Dataset exists: {data_path.exists()}")

## 4. Run the training pipeline

The training pipeline performs all steps from raw event-log loading to fitted models and validation results.

Its returned `TrainingPipelineResult` contains the trained models, aligned feature matrices, labels, case IDs, selected Random Forest parameters, validation tables, and dataset metadata.


In [ ]:
training_result = run_training_pipeline(
    event_log_path=data_path,
    positive_final_activities=positive_final_activities,
    min_prefix=minimum_prefix_length,
    random_forest_scoring="f1",
)

print("Training pipeline completed.")

## 5. Dataset and split summary

Each row in the encoded datasets represents a prefix of an original process case. Prefixes are generated separately inside the training, validation, and test splits.


In [ ]:
dataset_summary = (
    pd.Series(training_result.dataset_summary, name="value")
    .rename_axis("property")
    .reset_index()
)

dataset_summary["property"] = (
    dataset_summary["property"]
    .str.replace("_", " ", regex=False)
    .str.capitalize()
)

dataset_summary

In [ ]:
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "prefix_rows": [
            training_result.X_train.shape[0],
            training_result.X_validation.shape[0],
            training_result.X_test.shape[0],
        ],
        "features": [
            training_result.X_train.shape[1],
            training_result.X_validation.shape[1],
            training_result.X_test.shape[1],
        ],
        "positive_prefixes": [
            int((training_result.y_train == 1).sum()),
            int((training_result.y_validation == 1).sum()),
            int((training_result.y_test == 1).sum()),
        ],
        "negative_prefixes": [
            int((training_result.y_train == 0).sum()),
            int((training_result.y_validation == 0).sum()),
            int((training_result.y_test == 0).sum()),
        ],
    }
)

split_summary

## 6. Feature consistency and removed features

The training split defines the feature space. Validation and test data are aligned to exactly the same columns.

Features matching configured leakage-related keywords are removed consistently from all splits before model training.


In [ ]:
assert (
    training_result.X_train.shape[1]
    == training_result.X_validation.shape[1]
    == training_result.X_test.shape[1]
    == len(training_result.feature_columns)
)

assert training_result.X_train.shape[0] == len(training_result.y_train)
assert training_result.X_validation.shape[0] == len(training_result.y_validation)
assert training_result.X_test.shape[0] == len(training_result.y_test)
assert training_result.X_test.shape[0] == len(training_result.case_ids_test)

print("Feature matrices, labels, and case IDs are aligned.")
print(f"Final number of features: {len(training_result.feature_columns)}")
print("Removed feature columns:")
training_result.removed_feature_columns

## 7. Random Forest model selection

Several Random Forest configurations are compared on the validation set. The configuration with the best validation F1 score is selected as the final black-box model.


In [ ]:
print("Selected parameters:")
training_result.best_random_forest_parameters

In [ ]:
training_result.random_forest_validation_results

## 8. Validation-set model comparison

The majority baseline is included as a trivial reference. Logistic Regression is the interpretable model, and Random Forest is the more complex model used for SHAP explanations.


In [ ]:
training_result.validation_comparison

## 9. Run the evaluation pipeline

The evaluation pipeline uses the already trained models and the held-out test data. It creates:

- final test metrics;
- structured prediction outputs;
- model comparison and ROC plots;
- a Random Forest confusion matrix;
- global SHAP explanations;
- local SHAP explanations for representative test prefixes.


In [ ]:
evaluation_result = run_evaluation_pipeline(
    training_result=training_result,
    output_dir=output_dir,
    generate_shap=True,
    shap_max_samples=300,
    shap_top_n=15,
    local_shap_max_display=10,
)

print("Evaluation pipeline completed.")

## 10. Final test-set results

The held-out test set is used only after model selection. The table reports accuracy, precision, recall, F1, ROC AUC, and Precision-Recall AUC for all three models.


In [ ]:
test_comparison = evaluation_result.test_comparison.copy()
test_comparison

In [ ]:
metric_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]

best_models = {
    metric: test_comparison.loc[
        test_comparison[metric].idxmax(),
        "model",
    ]
    for metric in metric_columns
    if test_comparison[metric].notna().any()
}

pd.Series(best_models, name="best model").rename_axis("metric").reset_index()

## 11. Structured prediction output

The prediction table contains the real prefix case IDs, true labels, predicted labels, probabilities, model metadata, split name, classification threshold, and sprint identifier.


In [ ]:
evaluation_result.predictions.head(10)

In [ ]:
prediction_counts = (
    evaluation_result.predictions
    .groupby("model")
    .agg(
        prediction_rows=("case_id", "size"),
        unique_prefixes=("case_id", "nunique"),
        mean_probability=("probability", "mean"),
    )
    .reset_index()
)

prediction_counts

## 12. Evaluation plots

All figures are generated by the reusable evaluation pipeline and saved under `outputs/figures/`.


In [ ]:
for figure_name in [
    "model_comparison",
    "roc_curves",
    "random_forest_confusion_matrix",
]:
    figure_path = evaluation_result.figure_paths.get(figure_name)

    if figure_path is not None and Path(figure_path).exists():
        display(Markdown(f"### {figure_name.replace('_', ' ').title()}"))
        display(Image(filename=str(figure_path)))

## 13. Global SHAP explanations

The global SHAP summary and importance plots show which encoded features influence Random Forest predictions across the selected test sample.

The importance table ranks features by their mean absolute SHAP value. This measures influence magnitude, not whether a feature always pushes predictions toward the positive or negative class.


In [ ]:
for figure_name in [
    "shap_summary",
    "shap_importance",
]:
    figure_path = evaluation_result.figure_paths.get(figure_name)

    if figure_path is not None and Path(figure_path).exists():
        display(Markdown(f"### {figure_name.replace('_', ' ').title()}"))
        display(Image(filename=str(figure_path)))

In [ ]:
if evaluation_result.shap_importance is not None:
    display(evaluation_result.shap_importance.head(15))
else:
    print("SHAP generation was disabled.")

## 14. Local SHAP explanations

Representative local explanations are selected automatically:

- one correctly predicted positive prefix;
- one correctly predicted negative prefix;
- one misclassified prefix, when such a case exists.

The metadata table links each explanation to its real test-prefix ID and prediction.


In [ ]:
evaluation_result.local_explanations

In [ ]:
for explanation_type, figure_path in (
    evaluation_result.local_shap_paths.items()
):
    if Path(figure_path).exists():
        display(
            Markdown(
                f"### {explanation_type.replace('_', ' ').title()}"
            )
        )
        display(Image(filename=str(figure_path)))

## 15. Prototype prediction and explanation

The `predict_and_explain_case` function in `src/prototype/predict_explain.py` provides a
compact interface for making a prediction and returning the top SHAP contributors
for a single feature row. This is the prototype that a downstream application
could call to explain any individual prefix in real time.

The cell below takes the first test prefix, runs the Random Forest on it, and
returns a dictionary with the predicted class, the positive-class probability,
the top contributing features, and optionally a saved local SHAP bar plot.

In [ ]:
from src.prototype.predict_explain import predict_and_explain_case

# Use the first test prefix as a representative example.
sample_index = 0
sample_case_id = training_result.case_ids_test[sample_index]
sample_features = training_result.X_test[sample_index]

prototype_result = predict_and_explain_case(
    model=training_result.models["Random Forest"],
    case_features=sample_features,
    feature_columns=training_result.feature_columns,
    case_id=sample_case_id,
    top_n=5,
)

print(f"Case ID          : {prototype_result['case_id']}")
print(f"Predicted class  : {prototype_result['predicted_class']}")
print(f"Probability      : {prototype_result['prediction_probability']:.4f}")
print()
print("Top SHAP features:")
top_features_df = pd.DataFrame(prototype_result["top_features"])
display(top_features_df)

## 16. Generated artifacts

The evaluation pipeline records the locations of all generated reports and figures. These files can be used directly in project documentation and the final report.


In [ ]:
generated_reports = pd.DataFrame(
    [
        {"artifact": name, "path": str(path)}
        for name, path in evaluation_result.report_paths.items()
    ]
)

generated_reports

In [ ]:
generated_figures = pd.DataFrame(
    [
        {"artifact": name, "path": str(path)}
        for name, path in evaluation_result.figure_paths.items()
    ]
)

generated_figures

## 17. Final pipeline checks

These checks verify the most important integration properties of the final notebook run.


In [ ]:
expected_models = {
    "Majority baseline",
    "Logistic Regression",
    "Random Forest",
}

assert set(training_result.models) == expected_models
assert set(evaluation_result.test_comparison["model"]) == expected_models

expected_prediction_rows = (
    training_result.X_test.shape[0]
    * len(training_result.models)
)

assert len(evaluation_result.predictions) == expected_prediction_rows
assert evaluation_result.predictions["case_id"].notna().all()
assert evaluation_result.predictions["probability"].between(0, 1).all()
assert all(
    Path(path).exists()
    for path in evaluation_result.report_paths.values()
)
assert all(
    Path(path).exists()
    for path in evaluation_result.figure_paths.values()
)

print("All final pipeline checks passed.")

## Summary

This notebook demonstrates a complete prefix-level outcome-prediction workflow without implementing the workflow inside notebook cells.

The final implementation:

- defines outcomes from final lifecycle transitions;
- splits original cases temporally before prefix generation;
- keeps prefixes from the same original case in one data split;
- aligns train, validation, and test feature spaces;
- compares a majority baseline, Logistic Regression, and Random Forest;
- selects Random Forest hyperparameters using validation F1;
- evaluates all models once on the held-out test set;
- exports structured predictions with real prefix IDs;
- generates model-performance visualizations;
- provides global and local SHAP explanations;
- saves all reports and figures reproducibly.

The same underlying implementation can be executed either through this notebook or through:

```bash
python scripts/run_pipeline.py \
    --data data/raw/BPI_Challenge_2013_incidents/BPI_Challenge_2013_incidents.xes \
    --output-dir outputs
```

This removes the earlier dependency on running notebooks in a particular order and makes the project workflow reusable, testable, and reproducible.
